In [4]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix

pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 20)

In [5]:
# Load datasets
links = pd.read_csv('../data/full_dataset/links.csv')
movies = pd.read_csv('../data/full_dataset/movies.csv')
ratings = pd.read_csv('../data/full_dataset/ratings.csv')
tags = pd.read_csv('../data/full_dataset/tags.csv')

print('links', links.shape)
print('movies', movies.shape)
print('ratings', ratings.shape)
print('tags', tags.shape)

links (87585, 3)
movies (87585, 3)
ratings (32000204, 4)
tags (2000072, 4)


In [6]:
# Candidate filtering thresholds to reduce sparsity
user_thresholds = [10, 20, 30, 50]
movie_thresholds = [10, 20, 50, 100]

user_counts = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

def apply_thresholds(min_user, min_movie):
    eligible_users = user_counts[user_counts >= min_user].index
    eligible_movies = movie_counts[movie_counts >= min_movie].index
    filtered = ratings[
        ratings['userId'].isin(eligible_users) &
        ratings['movieId'].isin(eligible_movies)
    ]
    return filtered

stats = []
for u in user_thresholds:
    for m in movie_thresholds:
        f = apply_thresholds(u, m)
        stats.append({
            'min_user': u,
            'min_movie': m,
            'ratings_kept': len(f),
            'users_kept': f['userId'].nunique(),
            'movies_kept': f['movieId'].nunique(),
            'ratings_pct': len(f) / len(ratings)
        })

stats_df = pd.DataFrame(stats).sort_values(['ratings_pct', 'users_kept', 'movies_kept'], ascending=False)
stats_df.head(10)

,min_user,min_movie,ratings_kept,users_kept,movies_kept,ratings_pct
0,10,10,31842705,200948,31961,0.995078
4,20,10,31842705,200948,31961,0.995078
1,10,20,31725920,200948,23350,0.991429
5,20,20,31725920,200948,23350,0.991429
2,10,50,31498689,200947,16034,0.984328
6,20,50,31498689,200947,16034,0.984328
3,10,100,31228923,200947,12191,0.975898
7,20,100,31228923,200947,12191,0.975898
8,30,10,31025789,166757,31961,0.969550
9,30,20,30909680,166757,23350,0.965921


In [18]:
# Use fixed thresholds for balanced coverage vs speed
min_user = 20
min_movie = 50

print('Selected thresholds:', min_user, min_movie)
stats_df[(stats_df['min_user'] == min_user) & (stats_df['min_movie'] == min_movie)]

Selected thresholds: 20 50


,min_user,min_movie,ratings_kept,users_kept,movies_kept,ratings_pct
6,20,50,31498689,200947,16034,0.984328


In [19]:
# Filter ratings based on selected thresholds
eligible_users = user_counts[user_counts >= min_user].index
eligible_movies = movie_counts[movie_counts >= min_movie].index
ratings_f = ratings[
    ratings['userId'].isin(eligible_users) &
    ratings['movieId'].isin(eligible_movies)
].copy()

print('Filtered ratings shape:', ratings_f.shape)
print('Users kept:', ratings_f['userId'].nunique())
print('Movies kept:', ratings_f['movieId'].nunique())

Filtered ratings shape: (31498689, 4)
Users kept: 200947
Movies kept: 16034


In [20]:
# Create dense indices for users and movies
user_ids = ratings_f['userId'].unique()
movie_ids = ratings_f['movieId'].unique()

user_id_to_idx = {uid: i for i, uid in enumerate(user_ids)}
movie_id_to_idx = {mid: i for i, mid in enumerate(movie_ids)}

ratings_f['user_idx'] = ratings_f['userId'].map(user_id_to_idx)
ratings_f['movie_idx'] = ratings_f['movieId'].map(movie_id_to_idx)

print('Index ranges:', ratings_f['user_idx'].max(), ratings_f['movie_idx'].max())

Index ranges: 200946 16033


In [21]:
# Build sparse user-item matrix
n_users = ratings_f['user_idx'].nunique()
n_movies = ratings_f['movie_idx'].nunique()

X = csr_matrix(
    (ratings_f['rating'], (ratings_f['user_idx'], ratings_f['movie_idx'])),
    shape=(n_users, n_movies)
)

print('Sparse matrix shape:', X.shape)
print('Non-zeros:', X.nnz)

Sparse matrix shape: (200947, 16034)
Non-zeros: 31498689


In [22]:
# Time-based split (recommended for realistic evaluation)
ratings_f['timestamp'] = pd.to_datetime(ratings_f['timestamp'], unit='s')

cutoff = ratings_f['timestamp'].quantile(0.8)
train = ratings_f[ratings_f['timestamp'] <= cutoff]
test = ratings_f[ratings_f['timestamp'] > cutoff]

print('Train size:', train.shape)
print('Test size:', test.shape)
print('Cutoff:', cutoff)

Train size: (25198951, 6)
Test size: (6299738, 6)
Cutoff: 2018-08-17 22:48:46.800000


In [23]:
# Ensure output directory exists
from pathlib import Path
Path('../data/processed').mkdir(parents=True, exist_ok=True)

In [24]:
# Save artifacts for modeling
ratings_f.to_csv('../data/processed/ratings_filtered.csv', index=False)
train.to_csv('../data/processed/ratings_train.csv', index=False)
test.to_csv('../data/processed/ratings_test.csv', index=False)

pd.Series(user_id_to_idx).to_csv('../data/processed/user_id_to_idx.csv')
pd.Series(movie_id_to_idx).to_csv('../data/processed/movie_id_to_idx.csv')

print('Saved processed datasets to ../data/processed')

Saved processed datasets to ../data/processed
